# 基于 Triton 实现 MiniLLM Flash Attention

该文章的目标是在简单机器上用triton实现flash-attn，并且能够适配minillm模型。大致的目标如下：

1. 在 RTX 5090 上建立 Triton 的基本模型。
2. 写出一个 forward-only 的 causal flash-attention kernel。
3. 用可重复的测试验证 kernel 的数值正确性。
4. 适配到 MiniLLM `server_api.py` 推理路径中。

本文首先对triton做基本介绍，了解其基本概念，学习其如何编写加速算子，接着介绍falsh-attn，最后尝试使用triton实现falsh-attn并进行验证。

In [2]:
import inspect
import sys
import textwrap
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from minillm.model.config import MiniLLMConfig
from minillm.model.model_base import Attention as MiniAttention
from minillm.model.model_base import apply_rotary_pos_emb, precompute_freqs_cis, repeat_kv

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DEFAULT_DTYPE = torch.float16
DEFAULT_HEAD_DIM = 64

env_info = {
    "torch": torch.__version__,
    "triton": triton.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "capability": torch.cuda.get_device_capability(0),
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "repo_root": str(REPO_ROOT),
}
print(env_info)

/root/autodl-tmp/minillm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'torch': '2.10.0+cu128', 'triton': '3.6.0', 'gpu': 'NVIDIA GeForce RTX 5090', 'capability': (12, 0), 'bf16_supported': True, 'repo_root': '/root/autodl-tmp/minillm'}


## Triton 基本概念

triton是OpenAI提出的一个高性能算子编程框架，可以使用python语言编写，并通过LLVM后端编译为CUDA算子，它比 CUDA 更高层，但比纯 PyTorch 更接近硬件执行。

下列先了解triton的基本概念：

1. Program instance
- Triton 里一个 kernel 会启动很多个 program instance。
- 可以把它理解成“很多个并行执行的小程序”，每个小程序负责输出张量里的一个 tile。
- `tl.program_id(axis=0)` 和 `tl.program_id(axis=1)` 就是在拿当前 program 的二维坐标。

2. Tile / Block
- Triton 通常按块处理数据，例如 `BLOCK_M x BLOCK_N`。
- 这和 CUDA 里 thread block 的思路接近，但 Triton 暴露的是更偏张量视角的编程接口。
- 调参时最重要的问题通常是：一个 program 一次处理多少行、多少列、多少通道。

3. Pointer arithmetic
- Triton kernel 里大量代码都在做地址计算，例如 `base_ptr + row_offset * stride + col_offset`。
- 这一步决定了数据是如何从全局内存中被读出来的。


4. Masked load/store
- `tl.load(..., mask=..., other=0.0)` 和 `tl.store(..., mask=...)` 用来处理边界块。
- 因为张量尺寸通常不是 block size 的整数倍，所以最后一个 block 往往只覆盖一部分有效元素。

5. `tl.constexpr`
- `BLOCK_M`、`BLOCK_N`、`head_dim` 被声明成 `tl.constexpr`，表示它们在编译期就是常量。
- 这样 Triton 编译器才能更积极地做循环展开、寄存器分配和代码特化。

6. Grid
- Python 包装层里的 `grid = (...)` 定义了总共要启动多少个 program instance。
- 本 notebook 里 attention 的 grid 设计是：
  - `axis=0` 负责 query 方向的 block。
  - `axis=1` 负责 `batch * heads`。

如何理解Triton 和 CUDA 的关系：
- Triton 不是替代 CUDA 的“新硬件层”，而是构建在 GPU 编译栈之上的高层 kernel DSL。
- 它适合写规则、密集、张量化的 kernel，尤其适合 deep learning operator。
- 当算子逻辑很规则时，Triton 通常比手写 CUDA 更快迭代，也更容易做实验。

In [3]:
default_cfg = MiniLLMConfig(hidden_size=512, num_attention_heads=8, num_key_value_heads=2)
print(
    {
        "hidden_size": default_cfg.hidden_size,
        "num_attention_heads": default_cfg.num_attention_heads,
        "num_key_value_heads": default_cfg.num_key_value_heads,
        "head_dim": default_cfg.hidden_size // default_cfg.num_attention_heads,
        "flash_attn_flag": default_cfg.flash_attn,
    }
)

forward_src = inspect.getsource(MiniAttention.forward)
print(textwrap.dedent(forward_src))

{'hidden_size': 512, 'num_attention_heads': 8, 'num_key_value_heads': 2, 'head_dim': 64, 'flash_attn_flag': True}
def forward(
    self,
    x: torch.Tensor,
    position_embeddings: Tuple[torch.Tensor, torch.Tensor],  # 修改为接收cos和sin
    past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    use_cache=False,
    attention_mask: Optional[torch.Tensor] = None,
):
    bsz, seq_len, _ = x.shape
    xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
    xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
    xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
    xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)

    cos, sin = position_embeddings
    xq, xk = apply_rotary_pos_emb(xq, xk, cos[:seq_len], sin[:seq_len])

    # kv_cache实现
    if past_key_value is not None:
        xk = torch.cat([past_key_value[0], xk], dim=1)
        xv = torch.cat([past_key_value[1], xv], dim=1)
    past_kv = (xk, xv) if use_cache else None

   

In [ ]:
def build_causal_mask(seq_q: int, seq_k: int, device: torch.device | str) -> torch.Tensor:
    # 对 decode 场景，Q 往往小于 K。
    # 这里通过 k_idx <= q_idx + (K - Q) 来表达“当前 query 只能看到自己之前的 token”。
    q_idx = torch.arange(seq_q, device=device)[:, None]
    k_idx = torch.arange(seq_k, device=device)[None, :]
    return k_idx <= (q_idx + seq_k - seq_q)


def make_additive_causal_mask(
    seq_q: int,
    seq_k: int,
    device: torch.device | str,
    dtype: torch.dtype,
) -> torch.Tensor:
    # SDPA 既支持 bool mask，也支持 additive mask。
    # 这里统一构造成 additive mask，便于清楚表达 attention bias 的数学含义。
    keep = build_causal_mask(seq_q, seq_k, device).view(1, 1, seq_q, seq_k)
    mask = torch.zeros((1, 1, seq_q, seq_k), device=device, dtype=dtype)
    return mask.masked_fill(~keep, float("-inf"))


def reference_causal_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    scale = q.shape[-1] ** -0.5
    scores = torch.matmul(q.float(), k.transpose(-1, -2).float()) * scale
    keep = build_causal_mask(q.shape[-2], k.shape[-2], q.device)
    scores = scores.masked_fill(~keep.view(1, 1, q.shape[-2], k.shape[-2]), float("-inf"))
    probs = torch.softmax(scores, dim=-1).to(q.dtype)
    return torch.matmul(probs, v)


def sdpa_causal_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    # 用 PyTorch 官方高性能路径做第二个基线。
    # 如果 reference 与 SDPA 对不齐，说明不是 Triton 的问题，而是前置数学定义就写错了。
    additive_mask = make_additive_causal_mask(q.shape[-2], k.shape[-2], q.device, q.dtype)
    return F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=additive_mask,
        dropout_p=0.0,
        is_causal=False,
    )

In [5]:
def run_reference_attention_tests() -> pd.DataFrame:
    dtypes = [torch.float16]
    if torch.cuda.is_bf16_supported():
        dtypes.append(torch.bfloat16)

    rows = []
    for dtype in dtypes:
        for batch, heads, seq_q, seq_k, head_dim in [
            (2, 8, 64, 64, 64),
            (2, 8, 1, 257, 64),
            (1, 8, 37, 91, 64),
        ]:
            q = torch.randn(batch, heads, seq_q, head_dim, device=DEVICE, dtype=dtype)
            k = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
            v = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
            out_ref = reference_causal_attention(q, k, v)
            out_sdpa = sdpa_causal_attention(q, k, v)
            max_diff = (out_ref - out_sdpa).abs().max().item()
            rows.append(
                {
                    "dtype": str(dtype).replace("torch.", ""),
                    "shape": f"B={batch}, H={heads}, Q={seq_q}, K={seq_k}, D={head_dim}",
                    "max_diff": max_diff,
                }
            )
            tol = 2e-2 if dtype is torch.float16 else 3e-2
            assert max_diff < tol, (dtype, seq_q, seq_k, max_diff)

    return pd.DataFrame(rows)


reference_results = run_reference_attention_tests()
reference_results

,dtype,shape,max_diff
0,float16,"B=2, H=8, Q=64, K=64, D=64",0.001953
1,float16,"B=2, H=8, Q=1, K=257, D=64",0.000244
2,float16,"B=1, H=8, Q=37, K=91, D=64",0.000488
3,bfloat16,"B=2, H=8, Q=64, K=64, D=64",0.015625
4,bfloat16,"B=2, H=8, Q=1, K=257, D=64",0.001953
5,bfloat16,"B=1, H=8, Q=37, K=91, D=64",0.007812


In [6]:
@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    out_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def triton_add(x: torch.Tensor, y: torch.Tensor, block_size: int = 1024) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda
    assert x.shape == y.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(out.numel(), block_size),)
    add_kernel[grid](x, y, out, out.numel(), BLOCK_SIZE=block_size)
    return out

In [7]:
torch.manual_seed(0)
x = torch.rand(98_432, device=DEVICE, dtype=torch.float32)
y = torch.rand(98_432, device=DEVICE, dtype=torch.float32)
out_torch = x + y
out_triton = triton_add(x, y)
max_diff = (out_torch - out_triton).abs().max().item()
print({"warmup_add_max_diff": max_diff})
assert max_diff == 0.0

{'warmup_add_max_diff': 0.0}


## FlashAttention 原理与实现路径

这里实现的是一个适用于推理的 forward-only causal attention kernel，重点是理解 block-wise online softmax，而不是追求通用训练框架。

先看朴素 attention 的数学形式：

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}} + M\right)V
$$

其中：
- $Q \in \mathbb{R}^{B \times H \times Q \times D}$
- $K, V \in \mathbb{R}^{B \times H \times K \times D}$
- $M$ 是 causal mask 或 padding mask

朴素实现的问题不在于公式错，而在于中间张量太大：
- 会显式得到一个 `Q x K` 的 score 矩阵。
- 再对它做 softmax。
- 再乘以 `V`。
- 这会导致大量 HBM 读写，而长序列 attention 很容易被内存带宽卡住。

FlashAttention 的核心思想：
1. 不显式物化完整的 `QK^T`。
2. 只按块读取 `K/V`，每次处理一小块 score。
3. 在遍历块时维护 online softmax 的统计量，而不是先把全部 score 写回显存再统一 softmax。
4. 这样得到的数学结果与标准 attention 一致，但显存访问模式更高效。


实现工程路径是：
1. 先写 PyTorch reference，保证数学定义明确。
2. 再写 Triton warm-up kernel，确认环境与编译链工作正常。
3. 再实现 flash-attention forward kernel。
4. 再用 reference 做数值对齐。
5. 最后接到 MiniLLM attention 路径，做 prefill 和 decode smoke test。

注意事项：
1. `q` 的形状是 `[B, H, Q, D]`，`k/v` 的形状是 `[B, H, K, D]`。
2. 对于带 KV cache 的 decode，通常 `Q=1` 而 `K` 很大，所以 causal mask 不能简单写成方阵，而要用 `K - Q` 的偏移量来定义可见范围。
3. Triton kernel 里按 `BLOCK_M x BLOCK_N` 做块计算，并在遍历 key block 时维护 online softmax 的 `m_i` 与 `l_i`。
4. 这个版本只覆盖 `head_dim in {16, 32, 64, 128}`。对当前 MiniLLM，默认 `head_dim=64`，已经够用。


## Online Softmax 原理与公式推导

FlashAttention 的难点不在矩阵乘法，而在于：如果不显式保存完整的 score 矩阵，softmax 还能不能算对。

先看一行普通 softmax。设这一行 score 为 $x_1, x_2, \dots, x_n$，则

$$
\mathrm{softmax}(x_j) = \frac{e^{x_j}}{\sum_{t=1}^{n} e^{x_t}}
$$

为了数值稳定，通常会减去整行最大值 $m = \max_j x_j$：

$$
\mathrm{softmax}(x_j) = \frac{e^{x_j - m}}{\sum_{t=1}^{n} e^{x_t - m}}
$$

问题在于 FlashAttention 不能先把整行都读完再统一处理，因为那样又会回到显式物化完整 `QK^T` 的老路。

所以需要 online softmax。思路是：
- 把一整行 score 拆成多个 block。
- 每看到一个新 block，就更新“到目前为止的最大值”和“到目前为止的分母和”。
- 同时维护对 `V` 的加权累积结果。

设处理到第 $i$ 个 block 时：
- $m_i$ 表示目前见过的最大 score
- $l_i$ 表示以 $m_i$ 为基准后的指数和
- $acc_i$ 表示当前已经累计的输出分子

如果当前 block 的局部最大值是 $\tilde{m}$，局部分母是 $\tilde{l}$，则新的全局最大值是：

$$
m_{new} = \max(m_i, \tilde{m})
$$

旧分母和新分母都要重标定到 $m_{new}$ 这个共同基准上：

$$
l_{new} = l_i e^{m_i - m_{new}} + \tilde{l} e^{\tilde{m} - m_{new}}
$$

同理，输出分子的累积也要做同样的缩放：

$$
acc_{new} = acc_i e^{m_i - m_{new}} + \widetilde{acc} e^{\tilde{m} - m_{new}}
$$

最后一行输出就是：

$$
out = \frac{acc_{final}}{l_{final}}
$$

这就是 kernel 里 `m_i`、`l_i`、`acc` 的来源，也是下面这几行代码的数学意义：

```python
m_new = tl.maximum(m_i, m_ij)
alpha = tl.exp(m_i - m_new)
beta = tl.exp(m_ij - m_new)
acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v) * beta[:, None]
l_i = l_i * alpha + l_ij * beta
m_i = m_new
```

Online softmax 的关键是把已经处理过的 block 统计量和当前 block 的统计量统一重标定到同一个最大值基准上。这样即使一次只看到一部分 score，也能逐步维护全局 softmax 所需的分母和输出累积值。

In [14]:
@triton.jit
def flash_attn_fwd_kernel(
    q_ptr,
    k_ptr,
    v_ptr,
    o_ptr,
    stride_qb,
    stride_qm,
    stride_qd,
    stride_kb,
    stride_kn,
    stride_kd,
    stride_vb,
    stride_vn,
    stride_vd,
    stride_ob,
    stride_om,
    stride_od,
    q_len,
    k_len,
    scale,
    head_dim: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    # axis=0 负责 query 方向的 tile，axis=1 负责 batch * heads。
    pid_m = tl.program_id(axis=0)
    pid_b = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, head_dim)

    # 一次性读入一个 query tile，后续会拿它与多个 key block 做点积。
    q_ptrs = q_ptr + pid_b * stride_qb + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = (offs_m[:, None] < q_len) & (offs_d[None, :] < head_dim)
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    # online softmax 的三组状态：
    # m_i: 当前行见过的最大值
    # l_i: 以 m_i 为基准后的 softmax 分母
    # acc: 最终输出的分子累积值
    m_i = tl.full((BLOCK_M,), -float("inf"), tl.float32)
    l_i = tl.zeros((BLOCK_M,), tl.float32)
    acc = tl.zeros((BLOCK_M, head_dim), tl.float32)

    # 当 Q < K 时，decode 场景的 causal mask 需要向右偏移 K - Q。
    causal_shift = k_len - q_len

    for start_n in range(0, k_len, BLOCK_N):
        current_n = start_n + offs_n
        k_ptrs = k_ptr + pid_b * stride_kb + current_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        v_ptrs = v_ptr + pid_b * stride_vb + current_n[:, None] * stride_vn + offs_d[None, :] * stride_vd

        kv_mask = (current_n[:, None] < k_len) & (offs_d[None, :] < head_dim)
        k = tl.load(k_ptrs, mask=kv_mask, other=0.0)
        v = tl.load(v_ptrs, mask=kv_mask, other=0.0)

        # 当前 query tile 与当前 key block 的局部 score。
        qk = tl.dot(q, tl.trans(k)) * scale
        causal = current_n[None, :] <= (offs_m[:, None] + causal_shift)
        valid = (offs_m[:, None] < q_len) & (current_n[None, :] < k_len) & causal
        qk = tl.where(valid, qk, -float("inf"))

        # 先在当前 block 内求局部最大值与局部分母。
        m_ij = tl.where(offs_m < q_len, tl.max(qk, axis=1), 0.0)
        p = tl.where(valid, tl.exp(qk - m_ij[:, None]), 0.0)
        l_ij = tl.sum(p, axis=1)

        # 再把旧状态重标定到新的最大值 m_new 上，这就是 online softmax 的关键。
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        beta = tl.exp(m_ij - m_new)
        acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v) * beta[:, None]
        l_i = l_i * alpha + l_ij * beta
        m_i = m_new

    out = acc / tl.where(l_i > 0, l_i, 1.0)[:, None]
    o_ptrs = o_ptr + pid_b * stride_ob + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    o_mask = (offs_m[:, None] < q_len) & (offs_d[None, :] < head_dim)
    tl.store(o_ptrs, out.to(q.dtype), mask=o_mask)


def triton_flash_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    block_m: int = 64,
    block_n: int = 64,
) -> torch.Tensor:
    # 包装层负责三件事：
    # 1. 校验输入形状与 dtype
    # 2. 把 [B, H, T, D] 展平为 [B*H, T, D]
    # 3. 定义 grid 和 launch 参数
    assert q.ndim == 4 and k.ndim == 4 and v.ndim == 4
    assert q.is_cuda and k.is_cuda and v.is_cuda
    assert q.dtype == k.dtype == v.dtype
    assert q.shape[:2] == k.shape[:2] == v.shape[:2]
    assert k.shape[-2] == v.shape[-2]
    assert q.shape[-1] == k.shape[-1] == v.shape[-1]

    batch, heads, q_len, head_dim = q.shape
    _, _, k_len, _ = k.shape
    assert head_dim in {16, 32, 64, 128}

    q_flat = q.contiguous().view(batch * heads, q_len, head_dim)
    k_flat = k.contiguous().view(batch * heads, k_len, head_dim)
    v_flat = v.contiguous().view(batch * heads, k_len, head_dim)
    o_flat = torch.empty_like(q_flat)

    num_warps = 4 if head_dim <= 64 else 8
    grid = (triton.cdiv(q_len, block_m), batch * heads)
    flash_attn_fwd_kernel[grid](
        q_flat,
        k_flat,
        v_flat,
        o_flat,
        q_flat.stride(0),
        q_flat.stride(1),
        q_flat.stride(2),
        k_flat.stride(0),
        k_flat.stride(1),
        k_flat.stride(2),
        v_flat.stride(0),
        v_flat.stride(1),
        v_flat.stride(2),
        o_flat.stride(0),
        o_flat.stride(1),
        o_flat.stride(2),
        q_len,
        k_len,
        head_dim ** -0.5,
        head_dim=head_dim,
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        num_warps=num_warps,
        num_stages=2,
    )
    return o_flat.view(batch, heads, q_len, head_dim)

## 正确性测试

这一段只做一件事：把 Triton kernel 和我们上面定义的 PyTorch 参考实现逐项对齐。

测试覆盖：
- `prefill`: `Q = K`
- `decode`: `Q = 1, K >> 1`
- 非方阵 causal: `Q < K`
- `float16` 与 `bfloat16`

In [16]:
def run_triton_flash_tests() -> pd.DataFrame:
    dtypes = [torch.float16]
    if torch.cuda.is_bf16_supported():
        dtypes.append(torch.bfloat16)

    rows = []
    for dtype in dtypes:
        for batch, heads, seq_q, seq_k, head_dim in [
            (2, 8, 64, 64, 64),
            (2, 8, 1, 257, 64),
            (1, 8, 37, 91, 64),
            (1, 8, 128, 128, 64),
        ]:
            q = torch.randn(batch, heads, seq_q, head_dim, device=DEVICE, dtype=dtype)
            k = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
            v = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
            out_ref = reference_causal_attention(q, k, v)
            out_triton = triton_flash_attention(q, k, v)
            max_diff = (out_ref - out_triton).abs().max().item()
            rows.append(
                {
                    "dtype": str(dtype).replace("torch.", ""),
                    "shape": f"B={batch}, H={heads}, Q={seq_q}, K={seq_k}, D={head_dim}",
                    "max_diff": max_diff,
                }
            )
            tol = 2e-2 if dtype is torch.float16 else 3e-2
            assert max_diff < tol, (dtype, seq_q, seq_k, max_diff)

    return pd.DataFrame(rows)


flash_test_results = run_triton_flash_tests()
flash_test_results

,dtype,shape,max_diff
0,float16,"B=2, H=8, Q=64, K=64, D=64",0.001953
1,float16,"B=2, H=8, Q=1, K=257, D=64",0.000244
2,float16,"B=1, H=8, Q=37, K=91, D=64",0.000488
3,float16,"B=1, H=8, Q=128, K=128, D=64",0.001953
4,bfloat16,"B=2, H=8, Q=64, K=64, D=64",0.015625
5,bfloat16,"B=2, H=8, Q=1, K=257, D=64",0.001953
6,bfloat16,"B=1, H=8, Q=37, K=91, D=64",0.003906
7,bfloat16,"B=1, H=8, Q=128, K=128, D=64",0.015625


## 基准测试

这里的基准测试不是为了证明 Triton 一定比 PyTorch SDPA 快，而是为了回答一个更实际的问题：RTX 5090 上，这个自定义 kernel 的位置在哪里。

解释方式：
- `prefill-256` 和 `prefill-1024` 关注长序列前向。
- `decode-1x2048` 关注带 KV cache 的单步解码。
- 如果某些场景下 Triton 没赢，这也是正常结果，因为 PyTorch 2.10 自带的 SDPA 已经很强。

In [10]:
def benchmark_attention_suite(dtype: torch.dtype = torch.float16) -> pd.DataFrame:
    rows = []
    for name, batch, heads, seq_q, seq_k, head_dim in [
        ("prefill-256", 2, 8, 256, 256, 64),
        ("prefill-1024", 1, 8, 1024, 1024, 64),
        ("decode-1x2048", 2, 8, 1, 2048, 64),
    ]:
        q = torch.randn(batch, heads, seq_q, head_dim, device=DEVICE, dtype=dtype)
        k = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
        v = torch.randn(batch, heads, seq_k, head_dim, device=DEVICE, dtype=dtype)
        additive_mask = make_additive_causal_mask(seq_q, seq_k, DEVICE, dtype)

        ms_sdpa = triton.testing.do_bench(
            lambda: F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=additive_mask,
                dropout_p=0.0,
                is_causal=False,
            ),
            warmup=25,
            rep=100,
        )
        ms_triton = triton.testing.do_bench(
            lambda: triton_flash_attention(q, k, v),
            warmup=25,
            rep=100,
        )
        rows.append(
            {
                "case": name,
                "dtype": str(dtype).replace("torch.", ""),
                "sdpa_ms": round(ms_sdpa, 3),
                "triton_ms": round(ms_triton, 3),
                "speedup_vs_sdpa": round(ms_sdpa / ms_triton, 3),
            }
        )

    return pd.DataFrame(rows)


benchmark_results = benchmark_attention_suite()
benchmark_results

,case,dtype,sdpa_ms,triton_ms,speedup_vs_sdpa
0,prefill-256,float16,0.021,0.012,1.686
1,prefill-1024,float16,0.071,0.035,2.035
2,decode-1x2048,float16,0.123,0.064,1.920


## 适配到 MiniLLM 与 API Server

这节替换 `minillm/model/model_base.py` 里的 `Attention.forward`。

从工程视角看，接入路径应该分成四步：

1. 算子级验证
- 独立验证 Triton kernel 的数学正确性。
- 这一阶段不要急着碰模型代码，否则你很难判断错误到底在 kernel 还是在模型集成。

2. attention 级验证
- 按 MiniLLM 当前实现方式投影出 `q/k/v`。
- 应用 rotary embedding 与 KV cache。
- 用一个“安全基线”做参考，这里选择的是 PyTorch SDPA。

3. 路由级验证
- 当 `attention_mask` 全 1 时走 Triton；否则回退到安全的 PyTorch SDPA。
- 这一步的目标不是追求最广泛的兼容性，而是保证推理链路先稳定可控。

4. 端到端验证
- 真实调用 `model.generate(...)`。
- 观察吞吐、首 token 延迟、decode token/s，以及数值稳定性。

这里先在 notebook 中做一个“可验证的适配层”：
1. 先按 MiniLLM 当前实现投影出 `q/k/v`。
2. 应用 rotary embedding 与 KV cache。
3. 当 `attention_mask` 全 1 时走 Triton；否则回退到安全的 PyTorch SDPA。
4. 这一版只做 forward，已经足够覆盖推理。


In [15]:
def build_additive_mask(attention_mask: torch.Tensor, seq_q: int, seq_k: int, dtype: torch.dtype) -> torch.Tensor:
    # 把 padding mask 与 causal mask 合并成一个 additive mask。
    # 这样后续无论走 SDPA 还是回退路径，语义都保持一致。
    causal = build_causal_mask(seq_q, seq_k, attention_mask.device).view(1, 1, seq_q, seq_k)
    keep = causal & attention_mask[:, None, None, :].bool()
    mask = torch.zeros((attention_mask.shape[0], 1, seq_q, seq_k), device=attention_mask.device, dtype=dtype)
    return mask.masked_fill(~keep, float("-inf"))


def project_minillm_qkv(
    module: MiniAttention,
    x: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor],
    past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,
    use_cache: bool = False,
):
    # 这一层函数的意义是把“模型内部的 attention 前处理”单独拎出来。
    # 这样可以先验证 q/k/v 的投影、RoPE、KV cache 是否正确，再去比较不同算子后端。
    bsz, seq_len, _ = x.shape
    xq, xk, xv = module.q_proj(x), module.k_proj(x), module.v_proj(x)
    xq = xq.view(bsz, seq_len, module.n_local_heads, module.head_dim)
    xk = xk.view(bsz, seq_len, module.n_local_kv_heads, module.head_dim)
    xv = xv.view(bsz, seq_len, module.n_local_kv_heads, module.head_dim)

    cos, sin = position_embeddings
    xq, xk = apply_rotary_pos_emb(xq, xk, cos[:seq_len], sin[:seq_len])

    # 当前仓库里的 rotary 路径会把 q/k 提升到 float32。
    # 为了安全接到 SDPA 或 Triton，这里显式对齐到 value 的 dtype。
    xq = xq.to(xv.dtype)
    xk = xk.to(xv.dtype)

    if past_key_value is not None:
        # decode 场景把新的 k/v 接到历史 cache 后面。
        xk = torch.cat([past_key_value[0], xk], dim=1)
        xv = torch.cat([past_key_value[1], xv], dim=1)

    present = (xk, xv) if use_cache else None
    xq = xq.transpose(1, 2)
    xk = repeat_kv(xk, module.n_rep).transpose(1, 2)
    xv = repeat_kv(xv, module.n_rep).transpose(1, 2)
    return xq, xk, xv, present


def safe_sdpa_minillm_attention(
    module: MiniAttention,
    x: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor],
    past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,
    use_cache: bool = False,
    attention_mask: torch.Tensor | None = None,
):
    # 这是“安全基线”。
    # 集成 Triton 时，先和它对齐，再谈性能，不要反过来做。
    xq, xk, xv, present = project_minillm_qkv(module, x, position_embeddings, past_key_value, use_cache)
    seq_q, seq_k = xq.shape[-2], xk.shape[-2]
    additive_mask = None if attention_mask is None else build_additive_mask(attention_mask, seq_q, seq_k, xq.dtype)
    out = F.scaled_dot_product_attention(
        xq,
        xk,
        xv,
        attn_mask=additive_mask,
        dropout_p=0.0,
        is_causal=False,
    )
    out = out.transpose(1, 2).reshape(x.shape[0], x.shape[1], -1)
    return module.resid_dropout(module.o_proj(out)), present


def triton_minillm_attention(
    module: MiniAttention,
    x: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor],
    past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,
    use_cache: bool = False,
    attention_mask: torch.Tensor | None = None,
):
    # 路由策略非常简单：
    # 无 padding 的纯 causal 推理路径走 Triton；复杂 mask 场景回退到 SDPA。
    xq, xk, xv, present = project_minillm_qkv(module, x, position_embeddings, past_key_value, use_cache)
    if attention_mask is None or torch.all(attention_mask == 1):
        out = triton_flash_attention(xq, xk, xv)
    else:
        seq_q, seq_k = xq.shape[-2], xk.shape[-2]
        additive_mask = build_additive_mask(attention_mask, seq_q, seq_k, xq.dtype)
        out = F.scaled_dot_product_attention(
            xq,
            xk,
            xv,
            attn_mask=additive_mask,
            dropout_p=0.0,
            is_causal=False,
        )
    out = out.transpose(1, 2).reshape(x.shape[0], x.shape[1], -1)
    return module.resid_dropout(module.o_proj(out)), present

In [17]:
def run_minillm_integration_smoke_test(dtype: torch.dtype = torch.float16) -> pd.DataFrame:
    config = MiniLLMConfig(
        hidden_size=512,
        num_attention_heads=8,
        num_key_value_heads=2,
        max_position_embeddings=512,
    )
    module = MiniAttention(config).to(device=DEVICE, dtype=dtype).eval()
    cos, sin = precompute_freqs_cis(dim=config.hidden_size // config.num_attention_heads, end=512)
    cos = cos.to(DEVICE)
    sin = sin.to(DEVICE)

    rows = []
    with torch.no_grad():
        x = torch.randn(2, 32, config.hidden_size, device=DEVICE, dtype=dtype)
        mask = torch.ones(2, 32, device=DEVICE, dtype=torch.long)
        ref_out, ref_past = safe_sdpa_minillm_attention(
            module,
            x,
            (cos[:32], sin[:32]),
            use_cache=True,
            attention_mask=mask,
        )
        tri_out, tri_past = triton_minillm_attention(
            module,
            x,
            (cos[:32], sin[:32]),
            use_cache=True,
            attention_mask=mask,
        )
        prefill_diff = (ref_out - tri_out).abs().max().item()
        rows.append({"case": "prefill", "dtype": str(dtype).replace("torch.", ""), "max_diff": prefill_diff})
        assert torch.allclose(ref_out, tri_out, atol=2e-2, rtol=2e-2)

        x_next = torch.randn(2, 1, config.hidden_size, device=DEVICE, dtype=dtype)
        next_mask = torch.ones(2, 33, device=DEVICE, dtype=torch.long)
        ref_decode, _ = safe_sdpa_minillm_attention(
            module,
            x_next,
            (cos[32:33], sin[32:33]),
            past_key_value=ref_past,
            use_cache=True,
            attention_mask=next_mask,
        )
        tri_decode, _ = triton_minillm_attention(
            module,
            x_next,
            (cos[32:33], sin[32:33]),
            past_key_value=tri_past,
            use_cache=True,
            attention_mask=next_mask,
        )
        decode_diff = (ref_decode - tri_decode).abs().max().item()
        rows.append({"case": "decode", "dtype": str(dtype).replace("torch.", ""), "max_diff": decode_diff})
        assert torch.allclose(ref_decode, tri_decode, atol=2e-2, rtol=2e-2)

    return pd.DataFrame(rows)


integration_results = run_minillm_integration_smoke_test()
integration_results

,case,dtype,max_diff
0,prefill,float16,0.000122
1,decode,float16,0.000000


实验结果：

- 在 RTX 5090 上，这个 notebook 里测到 Triton 相对 SDPA 在几个测试场景上有约 `1.7x` 到 `2.0x` 的加速。

一些技术细节：

### 1. 为什么 FlashAttention 更快，但复杂度没有本质改变？

- attention 的主要计算项仍然和 $Q, K, D$ 相关，并没有神奇消失。
- FlashAttention 的核心收益来自 IO 优化，而不是把数学运算量降到更低阶。
- 它通过避免显式写回完整的 score / probability 矩阵，显著减少了 HBM 访存。

### 2. 为什么 online softmax 是对的？

- softmax 的关键依赖是整行最大值和整行分母和。
- online softmax 通过在每个 block 上维护 `m_i` 和 `l_i`，并把旧统计量重标定到新的最大值基准上，保证分块累计后的结果与整行一次性 softmax 等价。
- 所以它改变的是计算顺序，不改变数学结果。

### 3. 为什么 decode 场景比 prefill 更容易出错？

- 因为 decode 通常 `Q=1` 而 `K` 已经很长，causal mask 不是简单的左下三角方阵。
- 如果还沿用 `Q=K` 的遮罩逻辑，mask 会错位。
- 所以这里必须显式处理 `K - Q` 的偏移量。

### 4. 为什么要保留 PyTorch SDPA 回退路径？

- 自定义 kernel 应该先服务高价值主路径，而不是试图一次覆盖所有输入形态。
- 对复杂 `attention_mask`、非目标 `head_dim` 或未来扩展场景，保留 SDPA 回退路径可以降低系统风险。
- 这是工程上可维护、可回滚的做法。

### 5. 为什么集成时要把 `q/k` cast 回 `v.dtype`？

- 当前 MiniLLM 的 rotary 路径会把 `q/k` 提升到 float32，而 `v` 可能还是 float16 或 bfloat16。
- 高性能 attention 后端通常要求 `q/k/v` dtype 对齐。
- 所以这一步不是优化技巧，而是正确性前提。

### 6. Triton kernel 里最关键的三个参数是什么？

- `BLOCK_M`：一个 program 处理多少个 query。
- `BLOCK_N`：一次扫描多少个 key/value。
- `num_warps`：影响并行度与寄存器压力。
- 它们共同决定 occupancy、访存模式和寄存器使用，是调优重点。
